# Analysing a joint dataset

Step 3 of the ISO pipeline: an end-to-end joint analysis of the primary CMB (MFLike TT/TE/EE) and the
CMB-lensing reconstruction, fitting **exactly the products built upstream**:

1. build the joint likelihood on the smooth datasets from
   [`create_datasets.ipynb`](create_datasets.ipynb) and evaluate it at the fiducial point;
2. fold in the **cross-covariance** from
   [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb);
3. launch an MCMC.

We work at the `build_info` layer rather than `quickstart`, so the configuration stays an ordinary
Cobaya `info` dict with the dataset, accuracy and cosmology as explicit, editable knobs;
`resolve_aliases` recovers the role aliases (`.mflike`, `.lensing`). Because both smooth twins were
imprinted at the same ISO fiducial we fit at, the joint chi-square is **0 by construction**.

> **ISO-local overrides.** All three notebooks pass `defaults_dir="defaults"` to `build_info`, so the
> fiducial comes from the YAMLs in `ISO_sims/defaults/`: each file per-file *replaces* its packaged
> `soliket/presets/defaults/` counterpart, the rest fall back to the package. Here `cosmo.yaml`
> (cosmology, incl. the `mnu*` neutrino mass sum) and `theory.yaml` (the camb neutrino `extra_args`
> for the SO **2-eigenstate normal hierarchy**) are overridden; foreground and systematics fall back.
> The NH override is **camb-only** and needs no Python — `build_info` applies it.

## 1. A joint likelihood on the smooth datasets

`build_info("multigaussian")` returns the `info` for a single `MultiGaussianLikelihood` wiring
MFLike + CMB lensing onto one shared CAMB theory. We point each component at its smooth twin in
`sims/`: the MFLike `input_file` at `mflike_smooth.fits`, and the lensing `data_folder` at `sims/`
(its multi-hundred-MB `correction`/`fiducial` aux files are referenced by **absolute** path from the
shipped install, so we never copy them). `iso_multigaussian` centralises that wiring so the three
build points below stay identical.

In [ ]:
import os
from pathlib import Path

from cobaya.model import get_model
from cobaya.tools import resolve_packages_path

from soliket.presets import build_info, resolve_aliases

MGL = "soliket.gaussian.MultiGaussianLikelihood"
PACKAGES = resolve_packages_path()

# Inline artifact paths, matching create_datasets.ipynb / create_cross_covariance.ipynb.
SIMS = Path("sims")
MFLIKE_SMOOTH = SIMS / "mflike_smooth.fits"
LENSING_SMOOTH = SIMS / "lensing_smooth.sacc.fits"
XCOV = SIMS / "XCov_mflike_lensing.fits"
LENS_SHIPPED = os.path.join(PACKAGES, "data", "LensingLikelihood")


def loglike(model):
    """Total log-likelihood of a model at its fiducial point."""
    return float(sum(model.loglikes({})[0]))


def iso_multigaussian(sample=None, cross_cov_path=None):
    """Joint MFLike + lensing info at the ISO fiducial, wired to fit the sims/ twins."""
    info = build_info("multigaussian", sample=sample, defaults_dir="defaults")
    info["packages_path"] = PACKAGES
    opts = info["likelihood"][MGL]["options"]
    # MFLike component -> smooth CMB+fg data (absolute input_file; reuse shipped cov/Bbl).
    opts[0]["input_file"] = str(MFLIKE_SMOOTH.resolve())
    # Lensing component -> smooth twin in sims/; corrections + fiducial stay in the
    # shipped folder (absolute filenames bypass data_folder joining).
    opts[1]["data_folder"] = str(SIMS.resolve())
    opts[1]["data_filename"] = LENSING_SMOOTH.name
    opts[1]["correction_filename"] = os.path.join(
        LENS_SHIPPED, "corrections_lensing.sacc.fits"
    )
    opts[1]["fiducial_filename"] = os.path.join(
        LENS_SHIPPED, "fiducial_lensing.sacc.fits"
    )
    if cross_cov_path is not None:
        info["likelihood"][MGL]["cross_cov_path"] = str(cross_cov_path)
    # --- editable knobs ----------------------------------------------------
    # info["theory"]["camb"]["extra_args"]["lens_potential_accuracy"] = 4   # accuracy
    # -----------------------------------------------------------------------
    return info


info = iso_multigaussian()
model = get_model(info)
roles = resolve_aliases(model)

print("members :", type(roles.mflike).__name__, "+", type(roles.lensing).__name__)
print("joint loglike at fiducial:", round(loglike(model), 4), "(chi^2 = 0 twins)")

## 2. Adding the cross-covariance

Both probes look at the same sky, so their errors are correlated. The physical cross-covariance is the
`sims/XCov_mflike_lensing.fits` produced by [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb)
via `CrossCov.from_cmb_lensing(roles.mflike, roles.lensing)`. It keys the block by the components' real
names and carries their auto-covariances, so it drops straight into the likelihood through
`cross_cov_path`. We rebuild from the same wiring with that one extra option and compare the joint
log-likelihood with and without the cross term.

In [ ]:
if not XCOV.is_file():
    print("Cross-covariance not found:", XCOV)
    print("Run create_cross_covariance.ipynb (RUN_FULL = True) first to produce it.")
else:
    model_xcov = get_model(iso_multigaussian(cross_cov_path=XCOV))
    print("loglike without cross-cov:", round(loglike(model), 4))
    print("loglike with    cross-cov:", round(loglike(model_xcov), 4))

## 3. Running an MCMC

Sampling is Python-native. Passing `sample=[...]` to `build_info` turns the named dual parameters into
sampled ones (they get their priors back); add a `sampler` block and hand the `info` to `cobaya.run`.
The cell below sets up a short chain over `tau`, guarded by `RUN_MCMC` so the notebook runs
top-to-bottom without launching a multi-minute sampler.

In [ ]:
RUN_MCMC = False  # set True to launch the sampler (minutes)

info_mcmc = iso_multigaussian(
    sample=["tau"], cross_cov_path=XCOV if XCOV.is_file() else None
)

sampled = [
    p for p, v in info_mcmc["params"].items() if isinstance(v, dict) and "prior" in v
]
print("sampled parameters:", sampled)

if RUN_MCMC:
    from cobaya import run

    info_mcmc["sampler"] = {"mcmc": {"max_samples": 50, "Rminus1_stop": 0.1}}
    info_mcmc["output"] = "chains/multigaussian"
    updated_info, sampler = run(info_mcmc)
    print("done:", sampler.products()["sample"].shape)
else:
    print("RUN_MCMC is False - skipping the sampler.")

## Recap

- `iso_multigaussian()` gives the joint MFLike + CMB-lensing `info` at the ISO fiducial, wired to fit
  the smooth twins in `sims/`; `resolve_aliases(model)` recovers `.mflike` / `.lensing`.
- The cross-covariance from [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb) folds in
  through `cross_cov_path`.
- Sampling runs by adding a `sampler` block and calling `cobaya.run(info)`.

This closes the pipeline: [`create_datasets.ipynb`](create_datasets.ipynb) builds the data,
[`create_cross_covariance.ipynb`](create_cross_covariance.ipynb) the cross-covariance, and this
notebook the joint fit — all at one ISO fiducial, all through `sims/`. `quickstart("multigaussian")`
does the lazy one-call version when you want convenience over the knobs.